# Exploring the common voting space

Thin viewer over the pipeline outputs. Run `python -m political_compass.pipeline --chamber both` from the project root first, then execute cells top to bottom. (Run jupyter from the project root so `political_compass` imports.)

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from political_compass import viz

viz.apply_style()
CHAMBER = "House"  # or "Senate"
pos = pd.read_csv(ROOT / f"output/positions_{CHAMBER}.csv")
diag = pd.read_csv(ROOT / f"output/diagnostics_{CHAMBER}.csv", index_col=0)
print(f"{len(pos):,} member-congress positions, {pos['icpsr'].nunique():,} members, "
      f"congresses {pos['congress'].min()}\u2013{pos['congress'].max()}")
pos.head()

40,299 member-congress positions, 11,207 members, congresses 1–119


,congress,chamber,icpsr,dim1,dim2,cluster,bloc,n_votes
0,1,House,154,0.326728,2.295300,1,1,103
1,1,House,259,-0.328132,0.036993,0,0,68
2,1,House,379,-0.428218,0.250998,0,0,103
3,1,House,649,0.290157,2.342177,1,1,104
4,1,House,786,-1.528200,1.045847,0,0,28


In [2]:
def show_congress(t: int):
    """One congress in the common space, colored by bloc lineage."""
    sub = pos[pos["congress"] == t]
    colors = viz._bloc_colors(pos)
    fig, ax = plt.subplots(figsize=(6, 5))
    for b, g in sub.groupby("bloc"):
        ax.scatter(g["dim1"], g["dim2"], s=18, c=colors[int(b)], alpha=0.85,
                   linewidths=0.4, edgecolors=viz.SURFACE, label=f"bloc {b}")
    ax.set_title(f"{CHAMBER} \u00b7 Congress {t} \u00b7 {viz.first_year(t)}\u2013{viz.first_year(t) + 2}")
    ax.set_xlabel("dimension 1")
    ax.set_ylabel("dimension 2")
    ax.legend()
    plt.show()

show_congress(119)
show_congress(88)  # civil-rights era: watch the second dimension open up

C:\Users\alibe\AppData\Local\Temp\ipykernel_29820\2937216829.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alibe\AppData\Local\Temp\ipykernel_29820\2937216829.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Polarization over time: separation of the two k=2 clusters
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(diag.index, diag["separation_k2"], color=viz.LINE_SLOTS[0], lw=2)
ax.set_title(f"{CHAMBER}: two-cluster separation over time")
ax.set_xlabel("congress")
plt.show()

diag[["n_members", "n_rollcalls", "k", "silhouette_k2", "separation_k2",
      "n_anchors", "anchor_resid"]].describe().round(2)

C:\Users\alibe\AppData\Local\Temp\ipykernel_29820\2525199846.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,n_members,n_rollcalls,k,silhouette_k2,separation_k2,n_anchors,anchor_resid
count,119.00,119.00,119.00,119.00,119.00,118.00,118.00
mean,338.65,439.03,2.09,0.69,4.34,246.54,0.34
std,117.57,336.28,0.34,0.11,1.77,119.58,0.16
min,64.00,68.00,2.00,0.39,1.79,46.00,0.09
25%,236.00,156.00,2.00,0.61,3.04,126.25,0.22
50%,387.00,307.00,2.00,0.70,3.92,280.50,0.32
75%,441.00,655.00,2.00,0.77,5.17,359.50,0.42
max,452.00,1434.00,4.00,0.90,11.01,397.00,0.94


In [4]:
# Trajectory of any member: pick an icpsr from pos (long careers are most fun)
spans = pos.groupby("icpsr")["congress"].agg(["count", "min", "max"])
spans.sort_values("count", ascending=False).head(10)

,count,min,max
icpsr,,,
2605,30,84,113
10713,27,89,115
10075,27,77,103
9677,26,63,88
1611,25,68,92
14066,25,93,117
7232,24,71,94
10421,24,81,105
14873,23,97,119


In [5]:
def show_member(icpsr: int):
    g = pos[pos["icpsr"] == icpsr].sort_values("congress")
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(pos["dim1"], pos["dim2"], s=4, c=viz.MUTED, alpha=0.06, linewidths=0)
    ax.plot(g["dim1"], g["dim2"], color=viz.LINE_SLOTS[0], lw=2, marker="o", ms=4,
            mec=viz.SURFACE, mew=0.5)
    ax.annotate("start", (g.iloc[0]["dim1"], g.iloc[0]["dim2"]), color=viz.INK_2,
                textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.set_title(f"icpsr {icpsr}: {viz.first_year(g['congress'].min())}\u2013"
                 f"{viz.first_year(g['congress'].max()) + 2}")
    plt.show()

show_member(int(spans["count"].idxmax()))

C:\Users\alibe\AppData\Local\Temp\ipykernel_29820\1615780627.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Next steps** (deferred by design): join Voteview `HSall_members.csv` on `icpsr` to name members, color by real party, and orient the axes; add `HSall_rollcalls.csv` to interpret dimensions by issue area.